<div align="center">

<a href="https://colab.research.google.com/github/utkukose/human-computer-interaction-NB-lecture/blob/main/exams/midterm/MID3_website_navigation_and_form.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Midterm option 3: Navigation and a form of a website

**Human-Computer Interaction (BIL-429 İnsan-Bilgisayar Etkileşimi)**<br>Midterm project: A usability study of an interface. Starter Colab notebook

[Project page](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/exams/midterm) | [Course repository](https://github.com/utkukose/human-computer-interaction-NB-lecture)

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. First version: November 2026.</sub>

</div>

## How to use this starter Colab notebook

Save a copy in Drive first, with File, Save a copy in Drive. Every code cell runs as it is with an example case: Registering for a course on the website of a university. Replace the values marked `# TODO` with the measurements of your own interface. Then run all cells again from the top, with Runtime, Run all, and use the printed values and figures in the report. Each section names the week and the lecture section that explain its method. The last section collects the numbers for the results table of the report. Keep your own data in the cells, so that the Colab notebook documents how every number of the report was obtained.

## 0. Setup

The cell loads the libraries and creates the dictionary `RESULTS`, which collects the numbers of the report.

In [ ]:
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.spatial.distance import squareform

RANDOM_STATE = 42
RESULTS = {}            # the numbers for the results table of the report
print("Setup complete.")

## 1. Breadth, depth and scent of the menus

*Lecture: [Week 7, Breadth and depth](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/lecture.html#breadth-and-depth) and [Week 7, Information scent](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/lecture.html#information-scent)*

A menu tree has $d$ levels with $b_l$ labels each. At every level, a user reads labels and chooses one, which takes $a + c\,b_l$ seconds, and picks the right label with the probability $s_l$, the *scent* of the labels. The share of walks without a wrong turn is $P_0 = \prod_l s_l$, and the expected time is $E[T] = \sum_l (a + c\,b_l)/s_l$. Estimate $s_l$ from a first-click test: Ask users where they would click first, and count the right first clicks.

In [ ]:
A, C = 1.9, 0.11          # TODO: s per level and s per label (Week 7)
TREES = {   # TODO: (breadth, scent) of each level of each design
    "current, 3 levels of 4": [(4, 0.70), (4, 0.75), (4, 0.85)],
    "redesign, 2 levels of 8": [(8, 0.90), (8, 0.92)],
}

def tree_walk(levels, a=A, c=C):
    """P_0 and E[T] of a walk through a tree given as (breadth, scent)."""
    p0 = math.prod(sc for b, sc in levels)
    t = sum((a + c * b) / sc for b, sc in levels)
    return p0, t

for name, levels in TREES.items():
    p0, t = tree_walk(levels)
    pages = math.prod(b for b, sc in levels)
    print(f"{name}: {pages} pages, P_0 = {p0:.3f}, E[T] = {t:.2f} s")
    RESULTS[f"{name}: P_0"] = round(p0, 3)
    RESULTS[f"{name}: E[T] (s)"] = round(t, 2)

## 2. A card sort for the new menu

*Lecture: [Week 7, Card sorting](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/lecture.html#card-sorting)*

In an open card sort, each participant puts the pages of the website, written on cards, into groups of their own. The similarity of two cards is the share of the participants who put them together. Hierarchical clustering with average linkage joins the most similar cards first, and cutting the tree gives the groups of the new menu. The example simulates 15 participants; replace them with the sorts of at least 12 real users.

In [ ]:
CARDS = ["Course catalogue", "Register for a course", "Drop a course",
         "Timetable", "Exam dates", "Exam results", "Transcript",
         "Tuition fees", "Pay online", "Scholarships", "Library loans",
         "Room booking", "Student ID card", "Contact the registrar",
         "Academic calendar", "Course evaluation"]
# TODO: replace the simulated sorts by your participants' sorts, one list of
# group numbers per participant, in the order of CARDS
rng = np.random.default_rng(RANDOM_STATE)
LATENT = [0, 0, 0, 1, 1, 2, 2, 3, 3, 3, 4, 4, 4, 5, 1, 2]
SORTS = []
for _ in range(15):
    SORTS.append([g if rng.random() < 0.8 else int(rng.integers(0, 6))
                  for g in LATENT])

def similarity(sorts):
    """Share of the participants who put each pair of cards together."""
    S = np.zeros((len(CARDS), len(CARDS)))
    for sort in sorts:
        g = np.asarray(sort)
        S += g[:, None] == g[None, :]
    return S / len(sorts)

S = similarity(SORTS)
Z = linkage(squareform(1 - S, checks=False), method="average")
K = 6                    # TODO: the number of groups of the new menu
groups = fcluster(Z, K, criterion="maxclust")
for k in range(1, K + 1):
    print(f"group {k}:", ", ".join(c for c, g in zip(CARDS, groups) if g == k))
RESULTS["groups of the card sort"] = K

In [ ]:
# Figure: the dendrogram of the card sort
plt.figure(figsize=(9, 4))
dendrogram(Z, labels=CARDS, orientation="right", color_threshold=Z[-K + 1, 2])
plt.xlabel("distance 1 - similarity")
plt.tight_layout()
plt.show()

Name each group with a label that the participants would use, and check the labels with the first-click test of Section 1.

## 3. The form: Memory and defaults

*Lecture: [Week 3, Memory](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lecture.html#memory) and [Week 3, Decisions and defaults](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lecture.html#decisions-and-defaults)*

A field that asks for a code from memory fails more often the longer the code. The probability of recalling a code of length $L$ is $1/(1 + e^{(L - L_{50})/s})$, with a span $L_{50}$ of about 7 for plain digits. A pre-selected option makes a decision for everyone who keeps it. With $N$ uses, a share $q$ that keeps the default and a share $w$ for whom it is wrong, $W = N\,q\,w$ people get an unwanted setting.

In [ ]:
def p_recall(L, L50=7.0, s=0.8):
    """Probability of recalling a code of length L correctly."""
    return 1 / (1 + np.exp((np.asarray(L, float) - L50) / s))

FIELDS = {   # TODO: the codes that your form asks for, with their length
    "course code, typed": 7, "student number, typed": 9,
    "section number, typed": 2}
for name, L in FIELDS.items():
    print(f"{name}: L = {L}, recalled correctly {float(p_recall(L)):.3f}")
# TODO: N uses a day, share q that keeps the default, share w for whom it is wrong
DEFAULTS = {"first section of the course is pre-selected": (900, 0.85, 0.40)}
for name, (N, q, w) in DEFAULTS.items():
    print(f"{name}: W = {N} x {q} x {w} = {N * q * w:.0f} people a day")
    RESULTS[f"unwanted default: {name}"] = round(N * q * w)
worst = max(FIELDS, key=FIELDS.get)
RESULTS[f"recall of the {worst}"] = round(float(p_recall(FIELDS[worst])), 3)

## 4. A heuristic evaluation of the registration

*Lecture: [Week 5, Heuristic evaluation](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lecture.html#heuristic-evaluation) and [Week 5, How many evaluators](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lecture.html#how-many-evaluators)*

At least three evaluators inspect the registration alone against the ten heuristics, and the lists are then merged. Each evaluator rates the severity of every merged problem from 0 to 4. The adjustment of Lewis estimates the share of all problems that the evaluators found.

In [ ]:
PROBLEMS = [   # TODO: the merged problems with their heuristics
    ("H1", "No sign that a full course has a waiting list"),
    ("H3", "Registering cannot be undone for ten minutes"),
    ("H4", "Course and module mean the same thing"),
    ("H5", "Clashing courses can be registered together"),
    ("H6", "The course code must be typed from memory"),
    ("H9", "Errors list a code but not the course"),
]
FOUND = np.array([          # TODO: one row per evaluator
    [1, 0, 1, 1, 1, 0],
    [1, 1, 0, 1, 0, 1],
    [0, 1, 0, 1, 1, 1],
])
SEVERITY = np.array([       # TODO: severity from 0 to 4, one row per evaluator
    [2, 3, 1, 4, 3, 2],
    [2, 3, 1, 4, 2, 3],
    [1, 2, 2, 3, 3, 3],
])

def lewis_adjust(M):
    """Estimate of lambda from a 0/1 matrix and its adjustment for few rows."""
    M = np.asarray(M)
    M = M[:, M.sum(axis=0) > 0]
    n = M.shape[0]
    lam = M.mean()
    gt = np.mean(M.sum(axis=0) == 1)
    adj = 0.5 * (lam - 1 / n) * (1 - 1 / n) + 0.5 * lam / (1 + gt)
    return lam, gt, adj

lam, gt, adj = lewis_adjust(FOUND)
share = 1 - (1 - adj) ** FOUND.shape[0]
print(f"adjusted lambda {adj:.3f}: the evaluators found an estimated {share:.3f}")
ranked = pd.DataFrame(PROBLEMS, columns=["heuristic", "problem"])
ranked["mean severity"] = SEVERITY.mean(axis=0).round(2)
ranked = ranked.sort_values("mean severity", ascending=False)
print(ranked.to_string(index=False))
RESULTS["estimated share of the problems found"] = round(share, 3)
RESULTS["most severe problem"] = ranked.iloc[0]["problem"]

## 5. Results for the report

The cell collects the numbers that the sections above stored in `RESULTS`. Copy them into the results table of the report, and explain each one in the text. A value that changes the design deserves a figure.

In [ ]:
import pandas as pd

print(pd.Series(RESULTS, name="value", dtype=object).to_string())

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>